# Football Analytics — Match Analysis Example

This notebook demonstrates the core analytical workflow:
1. Load StatsBomb match data (no DB required — uses statsbombpy directly)
2. Compute key metrics: xG, passing accuracy, pressing intensity
3. Generate visualisations: shot map, passing network, xG timeline

**Dataset**: FIFA World Cup 2022 Final — Argentina vs France

---

In [ ]:
# Setup
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
from mplsoccer import Pitch, VerticalPitch
from statsbombpy import sb

pd.set_option("display.max_columns", 30)
print("Libraries loaded successfully")

## 1. Data Loading

StatsBomb provides free event-level data for selected competitions.
We'll use the 2022 World Cup Final as our example match.

In [ ]:
# Fetch World Cup 2022 matches
# Competition ID 43 = FIFA World Cup, Season ID 106 = 2022
matches = sb.matches(competition_id=43, season_id=106)
print(f"Total matches available: {len(matches)}")

# Find the Final
final = matches[matches["competition_stage"] == "Final"]
print(f"\nFinal: {final[['home_team', 'away_team', 'home_score', 'away_score']].to_string(index=False)}")

match_id = final["match_id"].iloc[0]
print(f"Match ID: {match_id}")

In [ ]:
# Fetch all events for the match
events = sb.events(match_id=match_id)
print(f"Total events: {len(events)}")
print(f"\nEvent types:\n{events['type'].value_counts().head(10)}")

## 2. Shot Analysis & xG

StatsBomb provides pre-computed xG (expected goals) for each shot.
xG quantifies shot quality based on location, angle, body part, and context.

**Key metric**: Total xG vs actual goals reveals over/under-performance.

In [ ]:
# Extract shots
shots = events[events["type"] == "Shot"].copy()

# Extract location coordinates
shots["location_x"] = shots["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
shots["location_y"] = shots["location"].apply(lambda x: x[1] if isinstance(x, list) else None)

# Rename xG column
if "shot_statsbomb_xg" in shots.columns:
    shots["xg"] = shots["shot_statsbomb_xg"]

# Summary by team
shot_summary = (
    shots.groupby("team")
    .agg(
        total_shots=("type", "count"),
        total_xg=("xg", "sum"),
        goals=("shot_outcome", lambda x: (x == "Goal").sum()),
        avg_xg_per_shot=("xg", "mean"),
    )
    .round(3)
)

print("=== Shot Summary ===")
print(shot_summary)
print("\nxG difference (goals - xG):")
print((shot_summary["goals"] - shot_summary["total_xg"]).round(2))

In [ ]:
# Shot Map — visualise shot locations and quality
pitch = VerticalPitch(half=True, pitch_type="statsbomb", line_color="#c7d5cc")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

teams = shots["team"].unique()
colors = ["#75AADB", "#FFFFFF"]  # Argentina light blue, France white
edge_colors = ["#2B5BA0", "#002654"]

for idx, (team, color, ec) in enumerate(zip(teams, colors, edge_colors)):
    ax = axes[idx]
    pitch.draw(ax=ax)
    team_shots = shots[shots["team"] == team]

    # Non-goals
    non_goals = team_shots[team_shots["shot_outcome"] != "Goal"]
    pitch.scatter(
        non_goals["location_x"],
        non_goals["location_y"],
        s=non_goals["xg"] * 500 + 30,
        c=color,
        edgecolors=ec,
        linewidth=1.2,
        alpha=0.7,
        ax=ax,
        zorder=2,
    )

    # Goals
    goals = team_shots[team_shots["shot_outcome"] == "Goal"]
    pitch.scatter(
        goals["location_x"],
        goals["location_y"],
        s=goals["xg"] * 500 + 50,
        c="red",
        edgecolors="black",
        linewidth=1.5,
        alpha=0.9,
        ax=ax,
        zorder=3,
        marker="*",
    )

    total_xg = team_shots["xg"].sum()
    goal_count = len(goals)
    ax.set_title(f"{team}\n{goal_count} goals | {total_xg:.2f} xG", fontsize=12, fontweight="bold")

plt.suptitle("Shot Map — World Cup 2022 Final", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/processed/shot_map_final.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Passing Analysis

Passing networks reveal team structure and key playmakers.
We analyse first-half completed passes between starters only.

In [ ]:
# Filter first-half completed passes for one team
team_name = teams[0]  # Argentina
passes = events[
    (events["type"] == "Pass")
    & (events["team"] == team_name)
    & (events["period"] == 1)
    & (events["pass_outcome"].isna())  # completed passes have no outcome
].copy()

passes["location_x"] = passes["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
passes["location_y"] = passes["location"].apply(lambda x: x[1] if isinstance(x, list) else None)
passes["end_x"] = passes["pass_end_location"].apply(lambda x: x[0] if isinstance(x, list) else None)
passes["end_y"] = passes["pass_end_location"].apply(lambda x: x[1] if isinstance(x, list) else None)

print(f"Completed passes (1st half, {team_name}): {len(passes)}")

# Average positions
avg_pos = (
    passes.groupby("player")
    .agg(avg_x=("location_x", "mean"), avg_y=("location_y", "mean"), passes_made=("type", "count"))
    .reset_index()
)

# Pass combinations
pass_combos = passes.groupby(["player", "pass_recipient"]).size().reset_index(name="pass_count")
pass_combos = pass_combos[pass_combos["pass_count"] >= 3]  # filter noise

print(f"\nTop pass combinations:\n{pass_combos.nlargest(5, 'pass_count').to_string(index=False)}")

In [ ]:
# Draw passing network
pitch = Pitch(pitch_type="statsbomb", line_color="#c7d5cc", pitch_color="#f0f0f0")
fig, ax = pitch.draw(figsize=(12, 8))

# Draw connections
max_count = pass_combos["pass_count"].max()
for _, row in pass_combos.iterrows():
    passer_pos = avg_pos[avg_pos["player"] == row["player"]]
    receiver_pos = avg_pos[avg_pos["player"] == row["pass_recipient"]]
    if passer_pos.empty or receiver_pos.empty:
        continue

    width = (row["pass_count"] / max_count) * 5 + 0.5
    alpha = min(row["pass_count"] / max_count + 0.3, 0.9)
    ax.plot(
        [passer_pos["avg_x"].iloc[0], receiver_pos["avg_x"].iloc[0]],
        [passer_pos["avg_y"].iloc[0], receiver_pos["avg_y"].iloc[0]],
        color="#1976D2",
        linewidth=width,
        alpha=alpha,
        zorder=1,
    )

# Draw nodes
node_sizes = (avg_pos["passes_made"] / avg_pos["passes_made"].max()) * 400 + 100
pitch.scatter(
    avg_pos["avg_x"], avg_pos["avg_y"], s=node_sizes, c="#FF5722", edgecolors="black", linewidth=1.5, ax=ax, zorder=2
)

# Labels (surname only)
for _, row in avg_pos.iterrows():
    name = row["player"].split()[-1]
    ax.annotate(
        name,
        (row["avg_x"], row["avg_y"]),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        fontweight="bold",
    )

ax.set_title(f"{team_name} — First Half Passing Network", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/processed/passing_network.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Pressing & Defensive Analysis

Pressing intensity reveals tactical approach: high press vs mid/low block.
We measure press actions by pitch zone and time period.

In [ ]:
# Pressing actions
pressures = events[events["type"] == "Pressure"].copy()
pressures["location_x"] = pressures["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
pressures["location_y"] = pressures["location"].apply(lambda x: x[1] if isinstance(x, list) else None)

# Zone classification
pressures["zone"] = pd.cut(
    pressures["location_x"], bins=[0, 40, 80, 120], labels=["Defensive Third", "Middle Third", "Attacking Third"]
)

press_summary = pressures.groupby(["team", "zone"]).size().unstack(fill_value=0)
press_summary["total"] = press_summary.sum(axis=1)
press_summary["high_press_pct"] = (press_summary["Attacking Third"] / press_summary["total"] * 100).round(1)

print("=== Pressing by Zone ===")
print(press_summary)

In [ ]:
# Pressing heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, team in enumerate(teams):
    ax = axes[idx]
    pitch = Pitch(pitch_type="statsbomb", line_color="#c7d5cc")
    pitch.draw(ax=ax)

    team_press = pressures[pressures["team"] == team]
    if not team_press.empty:
        pitch.kdeplot(
            team_press["location_x"], team_press["location_y"], ax=ax, cmap="YlOrRd", fill=True, levels=50, alpha=0.7
        )
    ax.set_title(f"{team} — Press Locations", fontsize=12, fontweight="bold")

plt.suptitle("Pressing Heatmaps — World Cup 2022 Final", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/processed/pressing_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. xG Timeline — Match Momentum

Cumulative xG over time shows which team dominated chance creation
and when momentum shifted — critical for post-match review.

In [ ]:
# Interactive xG timeline with Plotly
fig = go.Figure()

colors = {"Argentina": "#75AADB", "France": "#002654"}

for team in teams:
    team_shots_t = shots[shots["team"] == team].sort_values("minute")
    if team_shots_t.empty:
        continue

    cum_xg = team_shots_t["xg"].cumsum()
    minutes = [0] + team_shots_t["minute"].tolist()
    xg_vals = [0] + cum_xg.tolist()

    color = colors.get(team, "#333333")

    fig.add_trace(
        go.Scatter(
            x=minutes,
            y=xg_vals,
            mode="lines+markers",
            name=team,
            line=dict(color=color, width=2.5),
            marker=dict(size=7),
            hovertemplate="Min %{x}: Cum. xG = %{y:.2f}<extra></extra>",
        )
    )

# Add half-time and full-time markers
fig.add_vline(x=45, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=90, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_layout(
    title="Cumulative xG Timeline — World Cup 2022 Final",
    xaxis_title="Minute",
    yaxis_title="Cumulative xG",
    template="plotly_white",
    hovermode="x unified",
    width=900,
    height=450,
)
fig.show()

## 6. Key Findings

### Executive Summary (Hypothetical)

1. **Argentina dominated territorially** but France were clinical from fewer opportunities
2. **Pressing asymmetry**: Argentina pressed high (42% in attacking third) vs France's counter-pressing approach
3. **Key playmaker**: Messi's xG+xA contribution was the highest of any player in the match
4. **Set-piece danger**: 35% of France's xG came from set-piece situations — a key preparatory focus
5. **Second-half collapse**: Argentina's pressing intensity dropped 40% in the second half, correlating with France's comeback

---
*Analysis produced with StatsBomb open data. See technical appendix for methodology.*